# Modeling experimentation

As mentioned in the recommendation section of the README.md and Modeling.ipynb we are going to test a few models and a bit of hyperparameter tuning and some class balaancing. We will start with the models, then tune and finally then class balancing before running the models one more time to see if we can further improved the model.

We will test the data on XGBoost as it oftern acheives highers predictive accuracy than Random Forest on tabular data becuase it builds its trees sequentially, correcting the previous trees errors as it progresses though the modeling process. This can yeild a better ROC-AUC and more precise probability estimates. 

It is important to note that higher accuracy may not lead to a higher open rate for our client. The open rate achieved by targeting is determined by how well the model ranks customers. If Gradient boosting doe improve the ranking then reach for a given send volume will increase. So imporved model performance should lead to higher open rate among the targeted group, if all else remains the same.

Using exact same data and model as Modeling.ipynb notebook and adding XGBoost to compare side by side.

In [ ]:
# Add to venv is wishing to experiment
# !pip install xgboost

In [ ]:
!pip install mlflow

In [25]:
# New imports
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


#### Generate Synthetic Data and Target Variable

In [18]:
# Random see for reprocuibility
np.random.seed(42)

# Generating synthetic customer data
n_customers = 5000
data = {
    'customer_id': range(1, n_customers+1),
    'age' : np.random.randint(18, 70, n_customers),
    'income': np.random.normal(50000, 15000, n_customers).astype(int), #estimated
    'tenure': np.random.randint(1, 120, n_customers), #months
    'last_purchase_days': np.random.randint(1, 365, n_customers),
    'avg_order_value': np.random.normal(100, 30, n_customers).clip(20, 300).astype(int),
    'campaign_channel': np.random.choice(['email','social', 'push'], n_customers),
    'campaign_type': np.random.choice(['promotional', 'informational', 'loyalty'], n_customers),
    'sent': np.random.choice([0,1], n_customers, p=[0.2,0.8]), #80% were sent
}

df = pd.DataFrame(data)

In [19]:
# Generat target variable 'openeds' baseon on some hidden pattern
# Higher probability if: recent purchases, higher income, email channel, promotional type
def generate_opened(row):
    base_prob = 0.1
    if row['last_purchase_days'] > 60:
        base_prob += 2.0
    if row['income'] > 60000:
        base_prob += 1.0
    if row['campaign_channel'] == 'email':
        base_prob += 0.15
    if row['campaign_type'] == 'promotional':
        base_prob += 0.1
    # cap probability
    prob = min(base_prob, 0.9)
    return np.random.binomial(1, prob)

df['opened'] = df.apply(generate_opened, axis =1)

print("Data shape:", df.shape)
df.head()

Data shape: (5000, 10)


,customer_id,age,income,tenure,last_purchase_days,avg_order_value,campaign_channel,campaign_type,sent,opened
0,1,56,48353,96,249,95,email,loyalty,0,1
1,2,69,57462,92,74,146,push,promotional,1,1
2,3,46,44219,48,319,130,email,loyalty,1,1
3,4,32,56306,39,296,73,social,loyalty,1,1
4,5,60,37034,40,274,82,push,promotional,1,1


## Feature Engineering, Preprocessing and Modeling Pipelines

In [22]:
# Using only customers who were sent the campaign
train_df = df[df['sent']==1].copy()

# Features for the model
features = ['age', 'income', 'tenure', 'last_purchase_days', 'avg_order_value', 'campaign_channel', 'campaign_type']
X = train_df[features]
y = train_df['opened']

#  Spliting into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing: scale numeric, one-hot encode categorical
numeric_features = ['age', 'income', 'tenure', 'last_purchase_days', 'avg_order_value']
numeric_transformer = StandardScaler()
categorical_features = ['campaign_channel', 'campaign_type']
categorical_transformer = OneHotEncoder(drop='first', handle_unknown = 'ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

modelRF = Pipeline(steps= [
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

modelXG = Pipeline(steps= [
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42))
])

# Train with MLflow tracking
with mlflow.start_run(run_name="Random Forest with Preprocessing"):
    modelRF.fit(X_train, y_train)
    y_proba = modelRF.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(modelRF, "model")
    print(f"ROC-AUC RF: {auc}")

with mlflow.start_run(run_name="XGBoost"):
    modelXG.fit(X_train, y_train)
    y_proba = modelXG.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(modelXG, "model")
    print(f"ROC-AUC XG: {auc}")
    

2026/03/05 14:49:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 14:49:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 14:49:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/05 14:49:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

ROC-AUC RF: 0.781379487179487
ROC-AUC XG: 0.7818871794871796


## Performing GidSearchCV to find the best model and its parameters

MLflow can automaticaly log parameters and metrics for sklearn models including those from grid search cv. 

In [33]:
# Pipelines for each of the GridSearches

# Pipelines
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

In [34]:
# Parameter grids
# Parameter grid for Random Forest
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5, 10]   # corrected parameter name
}

# Parameter grid for XGBoost
param_grid_xgb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 6, 9],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0]
    # Note: XGBoost uses 'learning_rate' and 'subsample', not 'min_samples_split'
}

grid_search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)


In [44]:
from mlflow.tracking import MlflowClient

In [46]:
# Enable autologging so results are stored
mlflow.sklearn.autolog() #logs model params, metrics, model, sig( I/O schema), Artifacts ...
mlflow.xgboost.autolog() #logs model params, training/val metrics, feature importance, model itself

# Random Forest tuning
with mlflow.start_run(run_name="Random Forest Autolog Tuning") as rf_run:
    grid_search_rf.fit(X_train, y_train)
    best_model_rf = grid_search_rf.best_estimator_
    y_pred_proba_rf = best_model_rf.predict_proba(X_test)[:, 1]
    test_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
    mlflow.log_metric("test_roc_auc_rf", test_auc_rf)
    rf_run_id = rf_run.info.run_id

# XGBoost tuning
with mlflow.start_run(run_name="XGBoost Autolog Tuning") as xgb_run:
    grid_search_xgb.fit(X_train, y_train)
    best_model_xgb = grid_search_xgb.best_estimator_   # corrected: .best_estimator_
    y_pred_proba_xgb = best_model_xgb.predict_proba(X_test)[:, 1]
    test_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
    mlflow.log_metric("test_roc_auc_xgb", test_auc_xgb)
    xgb_run_id = xgb_run.info.run_id

Fitting 5 folds for each of 27 candidates, totalling 135 fits


2026/03/05 17:07:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 17:07:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 17:07:06 INFO mlflow.sklearn.utils: Logging the 5 best runs, 22 runs will be omitted.


Fitting 5 folds for each of 54 candidates, totalling 270 fits


2026/03/05 17:07:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 17:07:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/05 17:07:11 INFO mlflow.sklearn.utils: Logging the 5 best runs, 49 runs will be omitted.


In [55]:
# After searching for runs by name
rf_runs = client.search_runs(experiment_id, filter_string="tags.mlflow.runName = 'Random Forest Autolog Tuning'")
xgb_runs = client.search_runs(experiment_id, filter_string="tags.mlflow.runName = 'XGBoost Autolog Tuning'")

# Extract best run from each using the correct metric name
best_rf_run = max(rf_runs, key=lambda run: run.data.metrics.get("test_roc_auc_rf", 0))
best_xgb_run = max(xgb_runs, key=lambda run: run.data.metrics.get("test_roc_auc_xgb", 0))

print("=== Random Forest Best Run ===")
print(f"Run ID: {best_rf_run.info.run_id}")
print(f"Test ROC-AUC: {best_rf_run.data.metrics['test_roc_auc_rf']}")   # note the name
print("Best parameters:")
for param, value in best_rf_run.data.params.items():
    if param.startswith("classifier__"):
        print(f"  {param}: {value}")

print("\n=== XGBoost Best Run ===")
print(f"Run ID: {best_xgb_run.info.run_id}")
print(f"Test ROC-AUC: {best_xgb_run.data.metrics['test_roc_auc_xgb']}") # note the name
print("Best parameters:")
for param, value in best_xgb_run.data.params.items():
    if param.startswith("classifier__"):
        print(f"  {param}: {value}")

IndentationError: unexpected indent (1749975210.py, line 13)

## Further tuning
Additional XG parameters include 'classifier__min_child_weight', 'classifier__gamma', etc., to the XGBoost grid for more thorough tuning.

Consider using mlflow.xgboost.autolog() in addition to mlflow.sklearn.autolog() to capture XGBoost‑specific metrics like feature importance (though sklearn.autolog often works with XGBoost wrapped in a pipeline).